# 16 — PostgreSQL Persistence + Long-Term Memory + Optional pgvector

This notebook replaces the learning/dev persistence stack:

```text
SQLite checkpointer + InMemoryStore
```

with:

```text
Azure Database for PostgreSQL Flexible Server
│
├── PostgresSaver
│   → short-term thread / graph state
└── PostgresStore
    → long-term cross-thread memory
       └── optional pgvector → semantic retrieval
```

Sequence:

```text
16A Connect to Azure PostgreSQL
16B Short-term state with PostgresSaver
16C Long-term memory with PostgresStore
16D Saver + Store together in the Deep Agent
16E Durability + isolation validation
16F Optional pgvector semantic retrieval
```

We will not modify `src/` or Hosted Agent code here. First prove the behavior in a notebook; productionize afterward.

## Mental model

```text
                         Deep Agent
                            │
              ┌─────────────┴─────────────┐
              │                           │
              ▼                           ▼
        PostgresSaver                PostgresStore
              │                           │
          thread_id                user_id + namespace
              │                           │
     conversation / graph          durable cross-thread
     state / checkpoints           user memory
     interrupts
```

`thread_id` answers **which conversation?** We will create unique thread IDs with `uuid.uuid4()`.

`user_id` answers **whose durable memory?** For now there is one configured demo user. Later this must come from a trusted authenticated application/platform identity, never model-generated text.

# 16A — Connect to Azure PostgreSQL

You already provisioned PostgreSQL and added these values to `.env`:

```text
POSTGRES_HOST=<server>.postgres.database.azure.com
POSTGRES_DATABASE=deepagents
POSTGRES_USER=<your-Entra-UPN>
POSTGRES_SSLMODE=require
```

Use Microsoft Entra authentication via `DefaultAzureCredential`.

Install once from the repo root:

```powershell
uv add langgraph-checkpoint-postgres "psycopg[binary]" psycopg-pool
```

The notebook uses a fresh Entra token as the temporary PostgreSQL password. That is fine for learning, but not the final long-running Hosted Agent strategy because tokens expire.

In [41]:
import sys, os, uuid, inspect
import psycopg

from dotenv import load_dotenv, find_dotenv
from azure.identity import DefaultAzureCredential
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore

print("Python:", sys.version.split()[0])
print("psycopg:", getattr(psycopg, "__version__", "unknown"))
print("PostgresSaver:", PostgresSaver)
print("PostgresStore:", PostgresStore)

Python: 3.12.11
psycopg: 3.3.5
PostgresSaver: <class 'langgraph.checkpoint.postgres.PostgresSaver'>
PostgresStore: <class 'langgraph.store.postgres.base.PostgresStore'>


In [42]:
load_dotenv(find_dotenv())

POSTGRES_HOST = os.environ["POSTGRES_HOST"]
POSTGRES_DATABASE = os.environ["POSTGRES_DATABASE"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_SSLMODE = os.getenv("POSTGRES_SSLMODE", "require")

# Notebook-only user identity. Override with DEMO_USER_ID if desired.
USER_ID = os.getenv("DEMO_USER_ID", "local-demo-user")

# print("POSTGRES_HOST:", POSTGRES_HOST)
# print("POSTGRES_DATABASE:", POSTGRES_DATABASE)
# print("POSTGRES_USER:", POSTGRES_USER)
# print("POSTGRES_SSLMODE:", POSTGRES_SSLMODE)
# print("USER_ID:", USER_ID)

## Build a fresh Entra-authenticated PostgreSQL URI

The access-token scope is:

```text
https://ossrdbms-aad.database.windows.net/.default
```

Do not print the connection URI because it contains the bearer token.

In [43]:
from urllib.parse import quote

credential = DefaultAzureCredential()
POSTGRES_TOKEN_SCOPE = "https://ossrdbms-aad.database.windows.net/.default"

def build_postgres_uri() -> str:
    token = credential.get_token(POSTGRES_TOKEN_SCOPE).token
    encoded_user = quote(POSTGRES_USER, safe="")
    encoded_token = quote(token, safe="")
    return (
        f"postgresql://{encoded_user}:{encoded_token}"
        f"@{POSTGRES_HOST}:5432/{POSTGRES_DATABASE}"
        f"?sslmode={POSTGRES_SSLMODE}"
    )

print("Fresh PostgreSQL URI can be generated.")

Fresh PostgreSQL URI can be generated.


## Verify raw connectivity first

Before involving LangGraph, prove:

```text
networking ✓
Entra authentication ✓
database access ✓
SSL connection ✓
```

If this fails, fix PostgreSQL connectivity before debugging LangGraph.

In [44]:
with psycopg.connect(build_postgres_uri()) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT current_database(), current_user, version();"
        )
        database, current_user, version = cur.fetchone()

print("Database:", database)
print("Current user:", current_user)
print("Server:", version.split(",")[0])

Database: research_deepagent
Current user: shchitt_microsoft.com#EXT#@fdpo.onmicrosoft.com
Server: PostgreSQL 18.6 on x86_64-pc-linux-gnu


# 16B — Short-Term State with `PostgresSaver`

`PostgresSaver` is the PostgreSQL replacement for the SQLite checkpointer.

It owns thread/execution state such as messages, graph checkpoints, pending writes, and interrupts.

The application-facing identity is `thread_id`.

Do not write directly into LangGraph checkpoint tables.

In [5]:
print("PostgresSaver.from_conn_string:")
print(inspect.signature(PostgresSaver.from_conn_string))
print("\nPostgresSaver.setup:")
print(inspect.signature(PostgresSaver.setup))

PostgresSaver.from_conn_string:
(conn_string: 'str', *, pipeline: 'bool' = False) -> 'Iterator[PostgresSaver]'

PostgresSaver.setup:
(self) -> 'None'


In [6]:
with PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer:
    checkpointer.setup()

print("PostgresSaver schema initialized.")

PostgresSaver schema initialized.


In [7]:
def list_public_tables():
    with psycopg.connect(build_postgres_uri()) as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                ORDER BY table_name;
                '''
            )
            return [row[0] for row in cur.fetchall()]

tables_after_checkpointer = list_public_tables()
for table in tables_after_checkpointer:
    print(table)

checkpoint_blobs
checkpoint_migrations
checkpoint_writes
checkpoints


## Create a unique thread ID

Every independent conversation gets a unique, stable thread ID.

In [8]:
THREAD_ID = f"pg-thread-{uuid.uuid4()}"
thread_config = {"configurable": {"thread_id": THREAD_ID}}
print("THREAD_ID:", THREAD_ID)

THREAD_ID: pg-thread-4359db0b-57b8-4ebc-8c07-3e98f00ed35d


## Build the existing research agent with PostgreSQL checkpointing

The agent should depend on the generic checkpointer abstraction, so replacing SQLite with PostgreSQL should not require redesigning agent behavior.

In [9]:
from deep_agents_foundry import build_research_agent
from deep_agents_foundry.content import content_text

def final_text(result) -> str:
    messages = result.get("messages", [])
    return content_text(messages[-1]) if messages else ""

with PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer:
    agent = build_research_agent(checkpointer=checkpointer)
    first_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "For this thread only, remember the phrase "
                    "'blue checkpoint'. Reply briefly."
                ),
            }]
        },
        config=thread_config,
    )

print(final_text(first_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


Understood—I’ll remember the phrase **“blue checkpoint”** for this thread only.


## Same `thread_id` → continuity

In [10]:
with PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer:
    agent = build_research_agent(checkpointer=checkpointer)
    second_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": "What phrase did I ask you to remember in this thread?",
            }]
        },
        config=thread_config,
    )

print(final_text(second_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


You asked me to remember the phrase **“blue checkpoint.”**


## Different `thread_id` → short-term isolation

This is deliberately different from long-term memory.

In [11]:
ISOLATED_THREAD_ID = f"pg-thread-{uuid.uuid4()}"

with PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer:
    agent = build_research_agent(checkpointer=checkpointer)
    isolated_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "Without guessing, what special phrase did I mention "
                    "in another conversation thread?"
                ),
            }]
        },
        config={"configurable": {"thread_id": ISOLATED_THREAD_ID}},
    )

print("Original:", THREAD_ID)
print("Isolated:", ISOLATED_THREAD_ID)
print(final_text(isolated_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


Original: pg-thread-4359db0b-57b8-4ebc-8c07-3e98f00ed35d
Isolated: pg-thread-4a3f2484-8c7f-4dc5-8fb8-458eb11f9cd7
I can’t access your other conversation threads, so I have no way to know what phrase you mentioned there without you pasting it here (or describing it).


### 16B takeaway

```text
same thread_id      → continuity
different thread_id → isolation
```

“Short-term” describes the semantic scope of the state — a thread — not whether PostgreSQL survives a restart.

# 16C — Long-Term Memory with `PostgresStore`

Now switch mental models.

`PostgresStore` is not conversation history. It is durable application-owned memory such as:

```text
research preferences
stable user preferences
durable project facts
explicit remembered instructions
```

Its logical address is:

```text
namespace + key
```

For user memory, the namespace should include the trusted application `user_id`.

Example:

```text
("users", user_id, "research_preferences")
```

In [12]:
print("PostgresStore.from_conn_string:")
print(inspect.signature(PostgresStore.from_conn_string))
print("\nPostgresStore.setup:")
print(inspect.signature(PostgresStore.setup))
print("\nPostgresStore.search:")
print(inspect.signature(PostgresStore.search))

PostgresStore.from_conn_string:
(conn_string: 'str', *, pipeline: 'bool' = False, pool_config: 'PoolConfig | None' = None, index: 'PostgresIndexConfig | None' = None, ttl: 'TTLConfig | None' = None) -> 'Iterator[PostgresStore]'

PostgresStore.setup:
(self) -> 'None'

PostgresStore.search:
(self, namespace_prefix: 'tuple[str, ...]', /, *, query: 'str | None' = None, filter: 'dict[str, Any] | None' = None, limit: 'int' = 10, offset: 'int' = 0, refresh_ttl: 'bool | None' = None) -> 'list[SearchItem]'


In [13]:
with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    store.setup()

print("PostgresStore schema initialized.")

PostgresStore schema initialized.


## Inspect tables again

The same physical database now contains two logical persistence systems:

```text
PostgreSQL
├── checkpointer-owned tables
└── store-owned tables
```

In [14]:
tables_after_store = list_public_tables()
new_tables = sorted(set(tables_after_store) - set(tables_after_checkpointer))

print("Tables added after PostgresStore.setup():")
for table in new_tables:
    print(" -", table)

Tables added after PostgresStore.setup():
 - store
 - store_migrations


## Write a durable memory directly

Before involving the model, prove the Store contract deterministically.

In [15]:
MEMORY_NAMESPACE = ("users", USER_ID, "research_preferences")
MEMORY_KEY = f"preference-{uuid.uuid4()}"
MEMORY_VALUE = {
    "text": "Prefer architecture explanations with simple diagrams."
}

with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    store.put(MEMORY_NAMESPACE, MEMORY_KEY, MEMORY_VALUE)

print("Saved key:", MEMORY_KEY)
print("Namespace:", MEMORY_NAMESPACE)

Saved key: preference-4f2e321e-1fd3-4182-af4f-e5e1fd735f9f
Namespace: ('users', 'local-demo-user', 'research_preferences')


In [16]:
with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    item = store.get(MEMORY_NAMESPACE, MEMORY_KEY)

print("Retrieved:", item.value if item else None)

Retrieved: {'text': 'Prefer architecture explanations with simple diagrams.'}


## Namespace retrieval

At this stage semantic search is not required.

For small, structured memory collections, namespace retrieval is often the simplest and most deterministic design.

In [17]:
with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    memories = store.search(MEMORY_NAMESPACE, limit=20)

for memory in memories:
    print(memory.key, "→", memory.value)

preference-4f2e321e-1fd3-4182-af4f-e5e1fd735f9f → {'text': 'Prefer architecture explanations with simple diagrams.'}


### 16C takeaway

```text
PostgresStore
   ↓
namespace contains user_id
   ↓
cross-thread durable memory
```

And:

```text
long-term memory ≠ vector database
```

Vector search is only an optional retrieval strategy.

# 16D — Use `PostgresSaver` + `PostgresStore` Together

Now combine both persistence layers in the actual Deep Agent:

```text
                         Research Agent
                              │
                ┌─────────────┴─────────────┐
                │                           │
                ▼                           ▼
          PostgresSaver                PostgresStore
                │                           │
             thread_id                    user_id
                │                           │
        thread continuity           cross-thread memory
```

In [18]:
from deep_agents_foundry import ResearchContext

print("build_research_agent:")
print(inspect.signature(build_research_agent))
print("\nResearchContext:", ResearchContext)

build_research_agent:
(*, checkpointer=None, interrupt_on=None, store=None, skills=None)

ResearchContext: <class 'deep_agents_foundry.memory.ResearchContext'>


## Explicit memory write through the agent

The trusted `user_id` is passed through `ResearchContext`.

The model should never invent or select the user identity.

In [19]:
MEMORY_WRITE_THREAD_ID = f"pg-memory-thread-{uuid.uuid4()}"

with (
    PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer,
    PostgresStore.from_conn_string(build_postgres_uri()) as store,
):
    agent = build_research_agent(
        checkpointer=checkpointer,
        store=store,
    )

    memory_write_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "Please explicitly remember this durable research preference: "
                    "prefer primary Microsoft documentation when researching Azure technologies."
                ),
            }]
        },
        config={
            "configurable": {
                "thread_id": MEMORY_WRITE_THREAD_ID,
            }
        },
        context=ResearchContext(user_id=USER_ID),
    )

print(final_text(memory_write_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ResearchContext(user_id='local-demo-user'), input_type=ResearchContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-packages\p

Got it—I’ll prioritize primary Microsoft documentation (especially official sources like `learn.microsoft.com`) when researching Azure technologies.


## New thread + same user → long-term recall

This is the key distinction:

```text
new thread_id
same user_id
      ↓
new short-term conversation
same durable memory owner
```

In [20]:
MEMORY_RECALL_THREAD_ID = f"pg-memory-thread-{uuid.uuid4()}"

with (
    PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer,
    PostgresStore.from_conn_string(build_postgres_uri()) as store,
):
    agent = build_research_agent(
        checkpointer=checkpointer,
        store=store,
    )

    memory_recall_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "What durable research preferences have I asked you to remember? "
                    "Only report preferences actually available in memory."
                ),
            }]
        },
        config={
            "configurable": {
                "thread_id": MEMORY_RECALL_THREAD_ID,
            }
        },
        context=ResearchContext(user_id=USER_ID),
    )

print("Write thread:", MEMORY_WRITE_THREAD_ID)
print("Recall thread:", MEMORY_RECALL_THREAD_ID)
print(final_text(memory_recall_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ResearchContext(user_id='local-demo-user'), input_type=ResearchContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-packages\p

Write thread: pg-memory-thread-1bfbe04e-30a5-489d-9520-fd0f094d6e00
Recall thread: pg-memory-thread-2dc0bb9e-0a0b-410f-97c4-50d449a42adb
You have 1 saved durable research preference in memory:

- When researching **Azure technologies**, prefer **primary Microsoft documentation** (especially **learn.microsoft.com** and other official Microsoft sources) as the top sources.


## Temporary second-user isolation test

You currently have one real user in this project, but the storage design should already be multi-user safe.

We create a synthetic second user only for isolation testing.

In [21]:
OTHER_USER_ID = f"demo-other-user-{uuid.uuid4()}"
OTHER_USER_THREAD_ID = f"pg-memory-thread-{uuid.uuid4()}"

with (
    PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer,
    PostgresStore.from_conn_string(build_postgres_uri()) as store,
):
    agent = build_research_agent(
        checkpointer=checkpointer,
        store=store,
    )

    other_user_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "What durable research preferences are stored for me? "
                    "Do not guess."
                ),
            }]
        },
        config={
            "configurable": {
                "thread_id": OTHER_USER_THREAD_ID,
            }
        },
        context=ResearchContext(user_id=OTHER_USER_ID),
    )

print("Primary user:", USER_ID)
print("Temporary second user:", OTHER_USER_ID)
print(final_text(other_user_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ResearchContext(user_id='...45de-8a9b-396536124745'), input_type=ResearchContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\.venv\Lib\site-

Primary user: local-demo-user
Temporary second user: demo-other-user-1153ca3b-249a-45de-8a9b-396536124745
No durable research preferences are stored for you currently (memory is empty).


## Deterministic Store-level isolation check

Agent behavior is stochastic; storage isolation should also be checked directly.

In [22]:
primary_namespace = ("users", USER_ID, "research_preferences")
other_namespace = ("users", OTHER_USER_ID, "research_preferences")

with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    primary_items = store.search(primary_namespace, limit=100)
    other_items = store.search(other_namespace, limit=100)

print("Primary-user memory count:", len(primary_items))
print("Temporary second-user memory count:", len(other_items))

Primary-user memory count: 2
Temporary second-user memory count: 0


### 16D takeaway

Keep these identities separate:

```text
thread_id → which conversation?
user_id   → whose durable memory?
```

Later, Hosted Agent should derive them from separate trusted application/platform identities.

# 16E — Durability + Isolation Validation

PostgreSQL matters because state should outlive the Python process.

We want to prove:

```text
process lifecycle ≠ state lifecycle
```

The cleanest validation is a manual kernel restart.

## Print values to preserve before restart

Copy these values somewhere temporarily.

After restarting the kernel:

1. Re-run only the 16A import/config/connection cells.
2. Paste the old IDs into the validation cell.
3. Do not recreate the original messages or memories.
4. Verify PostgreSQL restores them.

In [23]:
print("COPY THESE BEFORE RESTART")
print("==========================")
print("THREAD_ID =", repr(THREAD_ID))
print("USER_ID =", repr(USER_ID))
print("MEMORY_NAMESPACE =", repr(MEMORY_NAMESPACE))
print("MEMORY_KEY =", repr(MEMORY_KEY))

COPY THESE BEFORE RESTART
THREAD_ID = 'pg-thread-4359db0b-57b8-4ebc-8c07-3e98f00ed35d'
USER_ID = 'local-demo-user'
MEMORY_NAMESPACE = ('users', 'local-demo-user', 'research_preferences')
MEMORY_KEY = 'preference-4f2e321e-1fd3-4182-af4f-e5e1fd735f9f'


## After restart: paste the old identifiers here

In [ ]:
# Example only — fill these AFTER restarting the kernel.
#
# RESTART_THREAD_ID = "pg-thread-..."
# RESTART_USER_ID = "local-demo-user"
# RESTART_MEMORY_NAMESPACE = (
#     "users",
#     RESTART_USER_ID,
#     "research_preferences",
# )
# RESTART_MEMORY_KEY = "preference-..."

In [8]:
RESTART_THREAD_ID = 'pg-thread-4359db0b-57b8-4ebc-8c07-3e98f00ed35d'
RESTART_USER_ID = 'local-demo-user'
RESTART_MEMORY_NAMESPACE = ('users', 'local-demo-user', 'research_preferences')
RESTART_MEMORY_KEY = 'preference-4f2e321e-1fd3-4182-af4f-e5e1fd735f9f'

## After restart: verify exact long-term memory

In [9]:
# Run only after defining RESTART_MEMORY_NAMESPACE and RESTART_MEMORY_KEY.

with PostgresStore.from_conn_string(build_postgres_uri()) as store:
    restarted_memory = store.get(
        RESTART_MEMORY_NAMESPACE,
        RESTART_MEMORY_KEY,
    )

print(restarted_memory.value if restarted_memory else None)

{'text': 'Prefer architecture explanations with simple diagrams.'}


## After restart: continue the old thread

In [11]:
from deep_agents_foundry import build_research_agent
from deep_agents_foundry.content import content_text

def final_text(result) -> str:
    messages = result.get("messages", [])
    return content_text(messages[-1]) if messages else ""

In [12]:
# Run only after defining RESTART_THREAD_ID.

with PostgresSaver.from_conn_string(build_postgres_uri()) as checkpointer:
    agent = build_research_agent(checkpointer=checkpointer)
    restarted_thread_result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": (
                    "What special phrase did I ask you to remember "
                    "earlier in this thread?"
                ),
            }]
        },
        config={
            "configurable": {
                "thread_id": RESTART_THREAD_ID,
            }
        },
    )

print(final_text(restarted_thread_result))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


You asked me to remember **“blue checkpoint.”**


## 16E validation matrix

| Test | Expected |
|---|---|
| Raw PostgreSQL connection | succeeds |
| `PostgresSaver.setup()` | checkpoint schema exists |
| same `thread_id` | history continues |
| different `thread_id` | short-term state isolated |
| `PostgresStore.setup()` | Store schema exists |
| exact memory lookup | succeeds |
| same `user_id`, new thread | durable memory available |
| different `user_id` | memory isolated |
| kernel/process restart | checkpoint remains |
| kernel/process restart | long-term memory remains |

If these pass, PostgreSQL has already replaced the *semantics* of SQLite + `InMemoryStore` in the notebook.

# 16F — Optional pgvector Semantic Retrieval

This section is optional.

You do **not** need pgvector to have durable long-term memory.

Without vectors:

```text
namespace / key / filter lookup
```

With vectors:

```text
natural-language query
      ↓
embedding
      ↓
vector similarity
      ↓
relevant memories
```

Semantic retrieval becomes useful when memory volume grows, wording varies, or the caller does not know the exact memory key/category.

## 16F.1 — Allow the PostgreSQL `vector` extension

Azure PostgreSQL Flexible Server must allow the `vector` extension before the database can create it.

This is an infrastructure/admin step. Do not run it automatically from the notebook.

First inspect the current allowlist:

```powershell
az postgres flexible-server parameter show `
  --resource-group <resource-group> `
  --server-name <server-name> `
  --name azure.extensions
```

Then add `vector` while preserving any extensions already configured.

Conceptually:

```powershell
az postgres flexible-server parameter set `
  --resource-group <resource-group> `
  --server-name <server-name> `
  --name azure.extensions `
  --value "<existing-extensions>,vector"
```

Do not blindly overwrite the existing extension list.

## 16F.2 — Create the extension in `deepagents`

Run this only after `vector` is allowlisted at the server level.

In [ ]:
# OPTIONAL
#
# with psycopg.connect(build_postgres_uri(), autocommit=True) as conn:
#     with conn.cursor() as cur:
#         cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
#
# print("vector extension created / already present.")

In [24]:
def vector_extension_info():
    with psycopg.connect(build_postgres_uri()) as conn:
        with conn.cursor() as cur:
            cur.execute(
                '''
                SELECT extname, extversion
                FROM pg_extension
                WHERE extname = 'vector';
                '''
            )
            return cur.fetchone()

vector_info = vector_extension_info()

if vector_info:
    print("pgvector available:", vector_info)
else:
    print(
        "vector extension is not installed in this database. "
        "That is fine unless you want to run semantic retrieval."
    )

vector extension is not installed in this database. That is fine unless you want to run semantic retrieval.


## 16F.3 — Configure an embedding model

For Azure OpenAI embeddings, configure an embedding deployment and environment values such as:

```text
AZURE_OPENAI_ENDPOINT=...
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=...
```

Install if needed:

```powershell
uv add langchain-openai
```

We continue to use Entra authentication rather than an API key.

The chat model and embedding model are separate deployments with different jobs.

In [ ]:
# OPTIONAL semantic-memory imports.

from azure.identity import get_bearer_token_provider
from langchain_openai import AzureOpenAIEmbeddings

embedding_token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint="os.environ["AZURE_OPENAI_ENDPOINT"]",
    azure_deployment=os.environ[
        "AZURE_OPENAI_EMBEDDING_DEPLOYMENT"
    ],
    azure_ad_token_provider=embedding_token_provider,
)



In [27]:
test_embedding = embeddings.embed_query("dimension check")
EMBEDDING_DIMS = len(test_embedding)

print("Embedding dimensions:", EMBEDDING_DIMS)

Embedding dimensions: 3072


## Why derive dimensions from the real deployment?

Vector columns require a fixed dimension.

Instead of copying a dimension from documentation, use one actual embedding and inspect:

```python
len(test_embedding)
```

That keeps the Store configuration aligned with the embedding function you are actually using.

## 16F.4 — Create a vector-enabled `PostgresStore`

The difference from the normal Store is the optional semantic index configuration:

```text
dims
embedding function
fields to index
```

Here we index only the `text` field.

In [29]:
# OPTIONAL: run after defining `embeddings` and `EMBEDDING_DIMS`.
#
# EMBEDDING_DIMS = 3072

semantic_index = {
    "dims": EMBEDDING_DIMS,
    "embed": embeddings,
    "fields": ["text"],
}

with PostgresStore.from_conn_string(
    build_postgres_uri(),
    index=semantic_index,
) as semantic_store:
    semantic_store.setup()

print("Vector-enabled PostgresStore initialized.")

AzureCliCredential.get_token failed: Failed to invoke the Azure CLI


CredentialUnavailableError: Failed to invoke the Azure CLI

## 16F.5 — Store semantic memories

Use a dedicated namespace for the demonstration.

In [ ]:
SEMANTIC_NAMESPACE = (
    "users",
    USER_ID,
    "semantic_research_preferences",
)

SEMANTIC_MEMORIES = {
    "architecture-style": {
        "text": "Prefer architecture explanations with simple diagrams."
    },
    "source-preference": {
        "text": (
            "Prefer primary Microsoft documentation when researching "
            "Azure technologies."
        )
    },
    "teaching-style": {
        "text": (
            "Start with an intuitive explanation before implementation details."
        )
    },
}

print("Semantic demo namespace:", SEMANTIC_NAMESPACE)
print("Prepared:", list(SEMANTIC_MEMORIES))

In [ ]:
# OPTIONAL: requires `semantic_index`.
#
# with PostgresStore.from_conn_string(
#     build_postgres_uri(),
#     index=semantic_index,
# ) as semantic_store:
#     for key, value in SEMANTIC_MEMORIES.items():
#         semantic_store.put(
#             SEMANTIC_NAMESPACE,
#             key,
#             value,
#         )
#
# print("Semantic memories written.")

## 16F.6 — Search using different wording

Stored:

```text
Prefer architecture explanations with simple diagrams.
```

Query:

```text
How should technical explanations be presented to me?
```

The words differ, but semantic retrieval should rank the relevant memory highly.

In [ ]:
# OPTIONAL: requires `semantic_index`.
#
# query = "How should technical explanations be presented to me?"
#
# with PostgresStore.from_conn_string(
#     build_postgres_uri(),
#     index=semantic_index,
# ) as semantic_store:
#     semantic_results = semantic_store.search(
#         SEMANTIC_NAMESPACE,
#         query=query,
#         limit=3,
#     )
#
# for result in semantic_results:
#     print("score:", getattr(result, "score", None))
#     print("key:", result.key)
#     print("value:", result.value)
#     print()

## 16F.7 — Semantic retrieval must still be user-scoped

Safe:

```text
trusted user_id
      ↓
user namespace
      ↓
semantic ranking inside that scope
```

Unsafe:

```text
semantic query
      ↓
all users' memories
```

A temporary second-user namespace can validate isolation.

In [ ]:
# OPTIONAL: requires `semantic_index`.
#
# semantic_other_user = f"demo-other-user-{uuid.uuid4()}"
# semantic_other_namespace = (
#     "users",
#     semantic_other_user,
#     "semantic_research_preferences",
# )
#
# with PostgresStore.from_conn_string(
#     build_postgres_uri(),
#     index=semantic_index,
# ) as semantic_store:
#     other_user_results = semantic_store.search(
#         semantic_other_namespace,
#         query="How should technical explanations be presented?",
#         limit=3,
#     )
#
# print("Other-user result count:", len(other_user_results))

## Exact retrieval vs semantic retrieval

| Situation | Start with |
|---|---|
| Small number of memories | namespace/key retrieval |
| Known preference categories | namespace/key retrieval |
| Deterministic retrieval desired | namespace/key retrieval |
| Many memories | consider semantic retrieval |
| User wording varies heavily | consider semantic retrieval |
| Relevant key is unknown | consider semantic retrieval |
| Need similarity ranking | pgvector |

A memory system does not become better merely because every memory has an embedding.

# Final Architecture

```text
                    Deep Agents / LangGraph
                             │
              ┌──────────────┴──────────────┐
              │                             │
              ▼                             ▼
      SHORT-TERM STATE               LONG-TERM MEMORY
              │                             │
       PostgresSaver                  PostgresStore
              │                             │
           thread_id                  user_id + namespace
              │                             │
       conversations                 preferences
       graph state                   durable facts
       interrupts                    explicit memories
                                            │
                                      optional only
                                            ↓
                                         pgvector
                                            │
                                     semantic recall

              └──────────────┬──────────────┘
                             ▼
             Azure Database for PostgreSQL
                   Flexible Server
```

One database can serve both systems, while the abstractions stay deliberately separate.

# Productionization Preview — Do Not Implement Yet

The notebook uses synchronous connections because they make the persistence model easier to understand.

The Hosted Agent should eventually move toward:

```text
Managed Identity / Entra identity
        ↓
refresh-aware PostgreSQL authentication
        ↓
connection pool
        ↓
AsyncPostgresSaver + AsyncPostgresStore
        ↓
Deep Agent
```

Important production concerns:

```text
one pool per process
not one pool per request

refreshable Entra credentials
not a frozen access token

setup/migrations at startup/deployment
not every request

thread_id from application conversation identity
user_id from authenticated platform/application identity

graceful pool shutdown
```

That later slice is what should fully remove SQLite from Hosted Agent runtime.

# Final Validation Checklist

```text
[ ] 16A raw PostgreSQL connection succeeds

[ ] 16B PostgresSaver.setup succeeds
[ ] same UUID thread continues
[ ] different UUID thread is isolated

[ ] 16C PostgresStore.setup succeeds
[ ] direct memory write/read succeeds
[ ] namespace retrieval succeeds

[ ] 16D agent works with Saver + Store
[ ] same user + new thread recalls durable preference
[ ] temporary second user is isolated

[ ] 16E restart kernel
[ ] old thread still continues
[ ] exact memory still exists

OPTIONAL:
[ ] Azure server allowlists vector
[ ] CREATE EXTENSION vector succeeds
[ ] embedding client works
[ ] semantic Store setup succeeds
[ ] differently-worded query finds relevant memory
[ ] semantic search remains user-scoped
```

Only after these pass should the implementation move into `src/`.

# Key Takeaways

1. **PostgreSQL is the physical persistence layer; Saver and Store are separate logical systems.**

```text
PostgresSaver → thread execution state
PostgresStore → long-term application memory
```

2. **`thread_id` and `user_id` solve different identity problems.**

```text
thread_id → which conversation?
user_id   → whose memory?
```

3. **Long-term memory does not require vectors.** Start with structured namespace retrieval.

4. **pgvector is a retrieval enhancement.** Add it when semantic similarity produces measurable value.

5. **Notebook auth is not final production auth.** A temporary Entra token is good for learning; Hosted Agent needs refresh-aware auth and pooling.

6. **Prove semantics first; productionize second.**

```text
Notebook
→ prove persistence behavior

src/
→ package reusable persistence factories

Hosted Agent
→ async + pooled + refresh-aware

SQLite
→ removed
```